In [1]:
import numpy as np

In [2]:
import pandas as pd
df= pd.read_csv("../Datasets/heart_disease.csv", na_values= "?").dropna()

In [3]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [4]:
df["target"]= (df["num"]>0).astype(int)
df["target"]

0      0
1      1
2      1
3      0
4      0
      ..
297    1
298    1
299    1
300    1
301    1
Name: target, Length: 297, dtype: int32

In [5]:
#assingning values
X= df.drop(columns= ["num", "target"])
y= df["target"]

print("rows:", len(X), "features:", X.shape[1])

rows: 297 features: 13


### 1. The problem: one train_test_split is unreliable
we can see with 2 model SVM and Random Forest

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size=0.2, random_state=10
)

#### SVM
start with SVM

In [7]:
from sklearn.svm import SVC
svm= SVC()

svm.fit(X_train, y_train)
svm.score(X_test, y_test)  #model score with unseen data

0.6666666666666666

### Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf= RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

0.8666666666666667

### KFold

In [9]:
from sklearn.model_selection import KFold

kf= KFold(n_splits=3)
kf

KFold(n_splits=3, random_state=None, shuffle=False)

In [10]:
data= [1, 2, 3, 4, 5, 6, 7, 8, 9]

for train_index, val_index in kf.split(data):
    
    print(train_index, val_index)#index values not actual data, every one gets chance

[3 4 5 6 7 8] [0 1 2]
[0 1 2 6 7 8] [3 4 5]
[0 1 2 3 4 5] [6 7 8]


In [11]:
#now we are supplying KFold to our variables and comparesing the scores
kf= KFold(n_splits=3)

svm_scores= []
rf_scores= []

#running a for loop for 3 time like we are specifyed above {kf= KFold(n_splits=3)}
for train_index, test_index in kf.split(X, y):
    X_train, X_test, y_train, y_test= X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    svm= SVC()
    svm.fit(X_train, y_train)
    svm_scores.append(svm.score(X_test, y_test))
    

    rf= RandomForestClassifier(n_estimators=40, random_state=42)
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))
    
    

In [12]:

from sklearn.model_selection import KFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

svm_scores = []
rf_scores = []

kf = KFold(n_splits=3)

for train_index, test_index in kf.split(X, y):

    # Split features and their matching target labels
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]  # Fixed

    # SVM
    svm = SVC()
    svm.fit(X_train, y_train)
    svm_scores.append(svm.score(X_test, y_test))

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=40,
        random_state=42
    )
    rf.fit(X_train, y_train)
    rf_scores.append(rf.score(X_test, y_test))

print("SVM scores:", svm_scores)
print("SVM mean accuracy:", sum(svm_scores) / len(svm_scores))

print("Random Forest scores:", rf_scores)
print("Random Forest mean accuracy:", sum(rf_scores) / len(rf_scores))


SVM scores: [0.6161616161616161, 0.6565656565656566, 0.6060606060606061]
SVM mean accuracy: 0.6262626262626263
Random Forest scores: [0.8383838383838383, 0.8080808080808081, 0.797979797979798]
Random Forest mean accuracy: 0.8148148148148149


### cross_val_score --> is usually the simplest way to perform cross-validation without manually writing a for loop.

#####  svm: The model to evaluate
#####  X:	All input features
#####  y:	All actual target labels
#####  cv=kf:	The cross-validation splitting strategy
#####  scoring="accuracy"	Measure the proportion of correct predictions

In [13]:
from sklearn.model_selection import cross_val_score

In [14]:
#for SVM
cross_val_score(svm, X, y, cv= kf)

array([0.61616162, 0.65656566, 0.60606061])

In [15]:
#for RandomForestClassifier
cross_val_score(rf, X, y, cv= kf)

array([0.83838384, 0.80808081, 0.7979798 ])

#### StratifiedKFold: 
is a cross-validation method that keeps the proportion of each class approximately the same in every fold.

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [16]:
from sklearn.model_selection import StratifiedKFold

skf= StratifiedKFold(n_splits=3)   #splits into 3 groups and each group proportion of each class approximately the same in every fold. 

numbers= [1, 2, 3, 4, 5, 6, 7, 8, 9]
labels= [0, 0, 0, 0, 0, 1, 1, 1, 1]

for train_index, test_index in skf.split(numbers, labels):
    print(train_index, test_index)

[2 3 4 6 7 8] [0 1 5]
[0 1 4 5 7 8] [2 3 6]
[0 1 2 3 5 6] [4 7 8]


#### GridSearchCV
It automatically tries different model settings, evaluates each using cross-validation, and identifies which settings perform best.

In [17]:
from sklearn import svm, datasets
iris= datasets.load_iris()
iris

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [18]:
import pandas as pd

df= pd.DataFrame(iris.data, columns= iris.feature_names)
df['flower']= iris.target
df['flower']= df['flower'].apply(lambda x: iris.target_names[x])
df[47:150]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
47,4.6,3.2,1.4,0.2,setosa
48,5.3,3.7,1.5,0.2,setosa
49,5.0,3.3,1.4,0.2,setosa
50,7.0,3.2,4.7,1.4,versicolor
51,6.4,3.2,4.5,1.5,versicolor
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [19]:
#SVM
from sklearn.model_selection import cross_val_score
cross_val_score(svm.SVC(kernel= 'linear', C= 10, gamma= 'auto'), iris.data, iris.target, cv= 5)

array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

##### you can see there are many things we need to do in order to get best outcome like changing input parameters, we can do mannually or run a loop 

In [20]:
#running a for loop   manual work

import numpy as np

kernels= ["rbf", "linear"]
C= [1, 10, 20]

avg_scores= {}   #later we store average scores here

for kval in kernels:
    for cval in C:
        cv_scores= cross_val_score(svm.SVC(kernel=kval, C= cval, gamma= 'auto'), iris.data, iris.target, cv= 5)
        avg_scores[kval + '_' + str(cval)] = np.average(cv_scores)


avg_scores     

{'rbf_1': 0.9800000000000001,
 'rbf_10': 0.9800000000000001,
 'rbf_20': 0.9666666666666668,
 'linear_1': 0.9800000000000001,
 'linear_10': 0.9733333333333334,
 'linear_20': 0.9666666666666666}

In [21]:
best_params = max(avg_scores, key=avg_scores.get)

print("Best combination:", best_params)
print("Best average accuracy:", avg_scores[best_params])

Best combination: rbf_1
Best average accuracy: 0.9800000000000001


#### To over come this manual work we have Grid Search CV 

In [22]:
#grid search cv
from sklearn.model_selection import GridSearchCV

clf= GridSearchCV(svm.SVC(gamma= "auto"), {
    'C': [1, 10, 20], 
    'kernel': ["rbf", "linear"]
}, cv= 5, return_train_score= False)


clf.fit(iris.data, iris.target)
clf.cv_results_

{'mean_fit_time': array([0.00210037, 0.00152988, 0.00184159, 0.00140004, 0.00163622,
        0.00150557]),
 'std_fit_time': array([0.00020639, 0.00044782, 0.00043166, 0.00048946, 0.00060752,
        0.00045028]),
 'mean_score_time': array([0.00148926, 0.00079474, 0.00120687, 0.00110326, 0.00141506,
        0.00119834]),
 'std_score_time': array([0.00060131, 0.0003976 , 0.0003878 , 0.00020226, 0.00052764,
        0.00040242]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
  {'C': 20, 'kernel': 'linear'}],


In [23]:
import pandas as pd

results = pd.DataFrame(clf.cv_results_)

results[[
    "param_C",
    "param_kernel",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values("rank_test_score")

,param_C,param_kernel,mean_test_score,std_test_score,rank_test_score
0,1,rbf,0.980000,0.016330,1
1,1,linear,0.980000,0.016330,1
2,10,rbf,0.980000,0.016330,1
3,10,linear,0.973333,0.038873,4
4,20,rbf,0.966667,0.036515,5
5,20,linear,0.966667,0.042164,6


In [24]:
print("Best parameters:", clf.best_params_)
print("Best CV accuracy:", clf.best_score_)

Best parameters: {'C': 1, 'kernel': 'rbf'}
Best CV accuracy: 0.9800000000000001


In [25]:
clf.best_estimator_

SVC(C=1, gamma='auto')

### RandomizedSearchCV
##### GridSearchCV:  when your search space is small and you want to test every specified combination.



##### RandomizedSearchCV:  when your search space is large and you want to set a fixed computation budget.



##### HalvingRandomSearchCV:  when you want to eliminate weaker candidates early and spend more resources on stronger ones.


### The trade-off: randomized search might miss the best combination because it doesn't try everything.

In [28]:
from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(
    svm.SVC(gamma="auto"),
    {
        "C": [1, 10, 20],
        "kernel": ["rbf", "linear"]
    },
    cv=5,
    return_train_score=False,
    n_iter=2,
    random_state=42
)

In [29]:
rs.fit(iris.data, iris.target)

print(rs.best_params_)
print(rs.best_score_)

{'kernel': 'rbf', 'C': 1}
0.9800000000000001


In [33]:
pd.DataFrame(rs.cv_results_)[["param_C", "param_kernel", "mean_test_score"]]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.98
1,1,linear,0.98


#### we can train multiliple model and their own hyperparameters also in RandomizedSearchCV
by passing multiple {} 

In [ ]:
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model_params= {
    "svm":{
        "model": svm.SVC(gamma= "auto"),
        "params":{
            "c":[1, 10, 20],
            "kernel": ["rbf", "linear"]
        }
    }, 

    "random_forest": {
        "model": RandomForestClassifier(random_state= 42),
        "params": {
            "n_estimators": [1, 5, 10]
        }
    }, 

    "logistic_regression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            "C": [1, 5, 10]
        }
    }
    
}